In [0]:
/***************************************  MODIFY: DECLARE ENVIRONMENT VARIABLE **********************************************/
/*----------- WARNING: Ensure to do a find and replace to also update commentary referring to the selected environment -----*/
DECLARE OR REPLACE env_var STRING DEFAULT "anz_dev";
/****************************************************************************************************************************/
EXECUTE IMMEDIATE

/*' ||env_var|| '*/
'CREATE OR REPLACE VIEW  ' ||env_var|| '.gld.vw_fact_wt AS
WITH fPCD as (
  select
    *
  from
    (
      select
        fPCD.*,
        tmb.Binder_DisplayGroup as Binder_Group,
        case
          when
            fPCD.DataSource = "IMS"
          then
            Row_Number() OVER (
                PARTITION BY  fPCD.QuoteID, fPCD.LineName, fPCD.LineKey   ---Transactionid
                ORDER BY
                  CASE
                    WHEN fPCD.QuoteOption = "bind" Then 0
                    WHEN fPCD.QuoteOption = "option1" Then 1
                    WHEN fPCD.QuoteOption = "option2" Then 2
                    WHEN fPCD.QuoteOption = "option3" Then 3
                    ELSE 4
                  END,
                  fPCD.isincluded desc,
                  fPCD.LineSubtype
              )
          else 1
        end as QuoteRowId
      from
        ' ||env_var|| '.brz.fact_PremiumCoreData fPCD
          left outer join ' ||env_var|| '.gld.vw_dim_binder tmb
            on tmb.Program_Id = fPCD.programid
      where
        /*fPCD.RiskID = "NZ00033728-000"
        And*/ (
          (fPCD.DataSource in ("IMS", "NZ BO", "eGlobal"))
          OR (
            fPCD.DataSource = "Trak"
            AND fPCD.BoundDate IS NOT NULL
          )
        )
        and fPCD.GreaterOfEffectiveOrBindDate >= "2024-01-01"  --ml jira-DMO-1448 
        and (
          fPCD.QuoteStatus not in ("Pending Cancellation", "Unbound Endorsement")
          or fPCD.QuoteStatus is null
        )
    ) fPCD
  where
    fPCD.QuoteRowId = 1
),
Renewing_Quote as (
  select
    *
  from
    (
      select
        Coalesce(BaseCurrencyOwnPremium, 0) as RenewingGrossWrittenPremiumAmount,
        Coalesce(TechPremium, 0) as RenewingTechPremiumAmount,
        Coalesce(RiskLimit, 0) as RenewingRiskLimit,
        Coalesce(Deductible, 0) as RenewingExcess,
        Coalesce(RevenueBasis, 0) as RenewingTurnover,
        ControlNo as RenewingControlNo,
        LineID,
        LineSubtype,
        PreviousPolicyNumber,
        QuoteStatus as RenewingQuoteStatus,
        QuoteID as RenewingQuoteId,
        ProducerCompanyName as RenewingBrokerName,
        UnderwriterNameFirstLast as RenewingUnderwriterName,
        SUM(
          case
            when
              QuoteStatus in ("Bound", "Processing Outstanding")
              and QuoteOption = "bind"
              and isincluded = true
            then
              Coalesce(BaseCurrencyOwnPremium, 0)
            else 0
          end
        ) OVER (
            PARTITION BY previouspolicynumber, LineId, Linesubtype
            order by startdate
          ) as TotalBoundRenewingGrossWrittenPremiumAmount,
        SUM(
          case
            when
              QuoteStatus in ("Cancelled", "Notice of Cancellation")
              and QuoteOption = "bind"
              and isincluded = true
            then
              Coalesce(BaseCurrencyOwnPremium, 0)
            else 0
          end
        ) OVER (
            PARTITION BY previouspolicynumber, LineId, Linesubtype
            order by startdate
          ) as TotalCancelledRenewingGrossWrittenPremiumAmount,
        RANK() OVER (
            PARTITION BY previouspolicynumber, LineId, Linesubtype
            order by QuoteID desc
          ) row_id,
        SUM(
          case
            when
              QuoteStatus in (
                "Bound", "Processing Outstanding", "Cancelled", "Notice of Cancellation"
              )
              and QuoteOption = "bind"
              and isincluded = true
            then
              Coalesce(BaseCurrencyOwnPremium, 0)
            else 0
          end
        ) OVER (
            PARTITION BY previouspolicynumber, LineId, Linesubtype
            order by startdate
          ) as TotalRenewingGrossWrittenPremiumAmount,
        SUM(
          case
            when
              QuoteStatus in (
                "Bound", "Processing Outstanding", "Cancelled", "Notice of Cancellation"
              )
              and QuoteOption = "bind"
              and isincluded = true
              and (
                TransactionTypeID Is Null
                OR TransactionTypeID = "ORIGNAL PREMIUM"
                OR TransactionTypeID = "C"
              )
            then
              Coalesce(BaseCurrencyOwnPremium, 0)
            else 0
          end
        ) OVER (
            PARTITION BY previouspolicynumber, LineId, Linesubtype
            order by startdate
          ) as TotalOriginalRenewingGrossWrittenPremiumAmount,
        SUM(
          case
            when
              QuoteStatus in (
                "Bound", "Processing Outstanding", "Cancelled", "Notice of Cancellation"
              )
              and QuoteOption = "bind"
              and isincluded = true
              and TransactionTypeID IN ("E", "N", "R")
            then
              Coalesce(BaseCurrencyOwnPremium, 0)
            else 0
          end
        ) OVER (
            PARTITION BY previouspolicynumber, LineId, Linesubtype
            order by startdate
          ) as TotalMTARenewingGrossWrittenPremiumAmount,
        SUM(
          case
            when
              QuoteStatus in (
                "Bound", "Processing Outstanding", "Cancelled", "Notice of Cancellation"
              )
              and QuoteOption = "bind"
              and isincluded = true
            then
              Coalesce(Coalesce(NewTotalCedeCommission, TotalCedeCommission), 0)
            else 0
          end
        ) OVER (
            PARTITION BY previouspolicynumber, LineId, Linesubtype
            order by startdate
          ) as TotalRenewingCedeCommissionAmount,
        SUM(
          case
            when
              QuoteStatus in (
                "Bound", "Processing Outstanding", "Cancelled", "Notice of Cancellation"
              )
              and QuoteOption = "bind"
              and isincluded = true
            then
              Coalesce(DualAdminFee, 0)
            else 0
          end
        ) OVER (
            PARTITION BY previouspolicynumber, LineId, Linesubtype
            order by startdate
          ) as TotalRenewingAdminFeeAmount,
        LatestQuotePerPolicy
      --LatestBoundQuotePerPolicy As LatestQuotePerPolicy
      from
        ' ||env_var|| '.brz.fact_PremiumCoreData
      where
        DataSource IN ("IMS")--- ml , "eGlobal"
        and LineID is not null
        and GreaterOfEffectiveOrBindDate >= "2024-01-01"  --ml jira-DMO-1448 
    ) RenQ
  where
    row_id = 1
    and RenewingQuoteId = LatestQuotePerPolicy
),
Policy_Totals as (
  select
    tpcd.controlno,
    tpcd.LineID,
    tpcd.LineSubtype,
    SUM(tpcd.BaseCurrencyOwnPremium) AS gwp,
    SUM(Coalesce(Coalesce(tPCD.NewTotalCedeCommission, tPCD.TotalCedeCommission), 0)) AS cede,
    SUM(tpcd.DualAdminFee) as adminFee,
    SUM(tpcd.ProducerCommissionAmount) as brokerage,
    SUM(Coalesce(Coalesce(tPCD.NewTotalCedeCommission, tPCD.TotalCedeCommission), 0))
      + SUM(tpcd.DualAdminFee) as income,
    SUM(
      CASE
        WHEN QuoteStatus in ("Bound", "Processing Outstanding") THEN tpcd.BaseCurrencyOwnPremium
        ELSE 0
      END
    ) as bound_gwp,
    SUM(
      CASE
        WHEN QuoteStatus in ("Cancelled", "Notice of Cancellation") THEN tpcd.BaseCurrencyOwnPremium
        ELSE 0
      END
    ) as cancelled_gwp
  FROM
    ' ||env_var|| '.brz.fact_PremiumCoreData tpcd
  WHERE
    tpcd.DataSource in ("IMS")
    and bounddate is not null
    and QuoteOption = "bind"
    and isincluded = true
    and tPCD.GreaterOfEffectiveOrBindDate >= "2024-01-01"  --ml jira-DMO-1448 
  GROUP BY
    tpcd.controlno,
    tpcd.LineID,
    tpcd.LineSubtype
),
Expiring_Totals as (
  select
    tpcd.RiskID,
    tpcd.LineID,
    tpcd.LineSubtype,
    SUM(Coalesce(tpcd.BaseCurrencyOwnPremium, 0)) AS gwp,
    SUM(Coalesce(Coalesce(tPCD.NewTotalCedeCommission, tPCD.TotalCedeCommission), 0)) AS cede,
    SUM(tpcd.DualAdminFee) as adminFee,
    SUM(tpcd.ProducerCommissionAmount) as brokerage,
    SUM(Coalesce(Coalesce(tPCD.NewTotalCedeCommission, tPCD.TotalCedeCommission), 0))
      + SUM(tpcd.DualAdminFee) as income,
    MAX(tpcd.ExpiryDate) as latestexpirydate
  FROM
    ' ||env_var|| '.brz.fact_PremiumCoreData tpcd
  WHERE
    tpcd.DataSource = "IMS"
    and bounddate is not null
    and QuoteOption = "bind"
    and isincluded = true
    and tPCD.GreaterOfEffectiveOrBindDate >= "2024-01-01"  --ml jira-DMO-1448 
  GROUP BY
    tpcd.RiskID,
    tpcd.LineID,
    tpcd.LineSubtype
),
fPCD_Allianz as (
  select
    *
  from
    (
      select
        *,
        row_number() over (
            partition by PreviousPolicyNumber, LineID, LineSubtype
            order by quoteid desc
          ) row_id
      from
        ' ||env_var|| '.brz.fact_PremiumCoreData fPCD_Allianz
      where
        SchemeName like "%Allianz%"
        and PolicyTypeID = 2
        and NOT EXISTS (
          SELECT
            1
          FROM
            ' ||env_var|| '.brz.fact_PremiumCoreData fPCD_expiring
          WHERE
            fPCD_expiring.RiskID = fPCD_Allianz.PreviousPolicyNumber
            AND fPCD_expiring.LineID = fPCD_Allianz.LineID
            AND fPCD_expiring.LineSubtype = fPCD_Allianz.LineSubtype
        )
    ) fPCD_Allianz
)
Select
  fPCD.DataSource As Data_Source,
  --fPCD.RiskID, Renewing_Quote.PreviousPolicyNumber,
  CASE
    WHEN fPCD.DataSource = "eGlobal" THEN fPCD.BoundDate
    ELSE fPCD.GreaterOfEffectiveOrBindDate
  END As Reporting_Date,
  fpCD.QuoteValidUntil as Quote_Valid_Until_Date,
  CAST(fPCD.WebRaterEntryDate as DATE) as Quote_Submission_Date,
  fPCD.PlacingBroker as Placing_Broker,
  fPCD.CurrencyCode as Currency_Code,
  fPCD.QuoteGUID as Quote_Guid,
  fPCD.QuoteOption as Quote_Option,
  Concat(
    Case fPCD.CompanyISOCountryCode
      When "AUS" Then "AU"
      Else "NZ"
    End,
    "|",
    fPCD.ProducerCompanyCode
  ) as Broker_Key,
  fPCD.SchemeName /* KK - Added this as per DMO-815 */,
  CASE
    WHEN fPCD.ProducerCompanyCode = "BIZN" THEN "BizCover"
    when UPPER(fPCD.linename) ilike "%MOTOR%" then "Motor"
    when
      UPPER(fPCD.linename) ilike "%MATERIAL DAMAGE%"
      or UPPER(fPCD.linename) ilike "%NATURAL DISASTER%"
      or UPPER(fPCD.linename) ilike "%BROKERWEB MDBI%"
      or fPCD.linename ilike "%Property%"
      or fPCD.linename ilike "%Contract Works%"
    then
      "Property"
    when
      UPPER(linename) ilike "%OTHER INSURANCE%"
      and fPCD.datasource = "eGlobal"
    then
      "Property"
    WHEN
      fPCD.DataSource = "IMS"
      and fPCD.CompanyISOCountryCode = "AUS"
      AND (
        fPCD.SchemeName like "Perrymans Project Managers%"
        OR fPCD.SchemeName IN ( /* KK - Added this as per DMO-805 */
       "PDA Professional Indemnity", "PDA Information Technology", "PDA Management Liability"
        ) 
      )
    THEN
      "Scheme"

    WHEN ClientName  ilike  "ACS Scheme%"  --("ACS Scheme - Schools","ACS Scheme - Churches")
    THEN
      "Scheme"
    WHEN
      (
        fPCD.QuoteInitiatedBy = "IMS"
        and fPCD.linename not like "%Property%"
      )
      AND Coalesce(fPCD.SchemeName, "") not in (
        "AON - Non Aligned Advisers", "AON Combined Advisor", "Penberthy", "Penberthy Platinum"
      ) 
    THEN
      "Specialty"
    WHEN
      (
        fPCD.QuoteInitiatedBy = "PORTAL"
        and fPCD.linename not like "%Property%"
      )
      AND Coalesce(fPCD.SchemeName, "") not in
       ( "AON - Non Aligned Advisers", "AON Combined Advisor", "Penberthy", "Penberthy Platinum")
    THEN
      "WebRater"
    WHEN fPCD.QuoteInitiatedBy = "BIZCOVER" THEN "BizCover"
    WHEN
      fPCD.QuoteInitiatedBy in ("Trak", "RiskWrite")
      and RWDC.DistributionChannel IS NOT NULL
    THEN
      REPLACE(RWDC.DistributionChannel, "WebRater - Rules", "WebRater")
    WHEN
      fPCD.QuoteInitiatedBy in ("Trak", "RiskWrite")
      and RWDC.DistributionChannel IS NULL
      and (
        fPCD.LineSubtype = "ACS Churches and Schools"
        OR fPCD.LineSubtype = "Professional Indemnity - MFAA Scheme"
        OR fPCD.ProducerCompanyCode = "AIPM"
      )
    THEN
      "Scheme"
    WHEN
      fPCD.DataSource = "IMS"
      and fPCD.CompanyISOCountryCode = "NZL"
      AND fPCD.SchemeName in (
        "AON - Non Aligned Advisers", "AON Combined Advisor", "Penberthy", "Penberthy Platinum"
      ) 
    THEN
      "Scheme"
    WHEN
      fPCD.SchemeName in (
        "Abacus - AON",
        "Abacus - Aon",
        "Advisorcover - MA Risk",
        "AMP Advisers - AON",
        "Aon-CEAS",
        "Architects and Graphic Designers - AIB",
        "ATF - Marsh",
        "Body Corporate - Crombie Lockwood",
        "Bookkeepers - AON",
        "Brokerweb - Willis",
        "Bureau - Churches Scheme",
        "Churches Scheme - Bureau",
        "CoffeeSure",
        "Counter Cover",
        "Cyber Offer",
        "EPA - Marsh",
        "Fabworx - CBA",
        "Ginger Group - AON",
        "Grocer Guard - Rothbury",
        "Harcourts - Crombie Lockwood",
        "HRINZ - Rothbury",
        "IAA & NZAMI - APEX",
        "ICIB Travel Agents - Marsh",
        "iiTP - I2I Brokers",
        "Kiwibank Contractors - i2ibrokers",
        "Law Plus - Willis",
        "Licensed Building Practitioners - Marsh",
        "Lockton - Inbroke",
        "LPMS - AON",
        "LPMS - Aon",
        "Marsh Accountants",
        "Marsh Big Accountants",
        "Marsh Small Accountants",
        "Mortgage Link - AON",
        "Non-aligned Advisers - Aon",
        "NorthSurance",
        "NZACS",
        "NZACS - Aon",
        "NZFMPF - Non Aligned - AON",
        "NZFMPS - AON",
        "NZFMSA",
        "NZFOA & NZFFA - AON Forests",
        "NZIBS - Willis",
        "Photographers - Rothbury",
        "Probus New Zealand Corporate Travel - Willis",
        "Project Arrow - Marsh",
        "PSC - Interflora",
        "Real Estate",
        "Real Estate - AON",
        "Real Estate - Penberthy",
        "Rebus New Zealand Corporate Travel",
        "Rebus New Zealand Corporate Travel - Willis",
        "REINZ - Crombie Lockwood",
        "Restaurants - PSC",
        "Roofers Association - Austinsure",
        "Sage Partners - Forestry",
        "Scheme",
        "Schools - Apex",
        "Valuers Inspectors - JLT",
        "Video Retailers",
        "Videoguard - Hutchinson Rodway",
        "Vision Sure",
        "AON - Non Aligned Advisers",
        "AON Combined Advisor"
      )
    THEN
      "Scheme"
    ELSE "Specialty"
  END As Segment,
  CAST(date_format(fPCD.BoundDate, "yyyy-MM-dd") AS DATE) As Bound_Date,
  CAST(date_format(fPCD.CreatedDate, "yyyy-MM-dd") AS DATE) As Sales_Date,
  fPCD.TransactionID As Transaction_Id,
  fPCD.ControlNo As Control_No,
  fPCD.RiskID As Policy_Number,
  fPCD.QuoteID As Quote_ID,
  concat(fPCD.QuoteGUID, "|", fPCD.lineproducttype) as Quote_Product_Key,
  fPCD.QuoteStatus As Quote_Status,
  fPCD.QuoteStatusReason As Quote_Status_Reason,
  CASE
    WHEN fPCD.PolicyTypeID = 1 Then "New Business"
    WHEN fPCD.policyTypeId = 2 THEN "Renewal"
    ELSE cast(fPCD.policyTypeId As String)
  END As Overarching_Policy_Type,
  Case
    When
      fPCD.DataSource = "IMS"
    Then
      Case
        When Coalesce(fPCD.CurrentlyInsuredWith, "") = "DUAL Renewal" Then "Renewal"
        Else "New Business"
      End
    When
      fPCD.DataSource = "Trak"
    Then
      CASE
        WHEN fPCD.PolicyTypeID = 1 Then "New Business"
        WHEN fPCD.PolicyTypeId = 2 THEN "Renewal"
        ELSE cast(fPCD.policyTypeId As String)
      END
    When
      fPCD.DataSource in ("NZ BO", "eGlobal")
    Then
      case
        when fPCD.PolicyTypeID = 1 Then "New Business"
        ELSE "Renewal"
      End
  End As Policy_Type,
  fPCD.PreviousPolicyNumber As Previous_Policy_Number,
  CASE
    WHEN
      CHARINDEX(";", fPCD.ClientName) > 0
    THEN
      LEFT(fPCD.ClientName, CHARINDEX(";", fPCD.ClientName) - 1)
    WHEN
      CHARINDEX(",", fPCD.ClientName) > 0
    THEN
      LEFT(fPCD.ClientName, CHARINDEX(",", fPCD.ClientName) - 1)
    ELSE ClientName
  END AS Client_Name,
  fPCD.InsuredGUID as Insured_Id,
  case
    when
      Quote_Status in ("Bound", "Processing Outstanding")
      and QuoteOption = "bind"
      and isincluded = true
    then
      concat_ws(
        "/",
        collect_set(
          case
            when
              QuoteStatus in ("Bound", "Processing Outstanding")
              and QuoteOption = "bind"
              and isincluded = true
            then
              mLOB.Line_Display_Name
            else null
          end
        ) OVER (PARTITION BY fPCD.InsuredGUID)
      )
    else NULL
  end as Overall_Product_Combination_Per_Insured,
  case
    when
      Quote_Status in ("Bound", "Processing Outstanding")
      and QuoteOption = "bind"
      and isincluded = true
    then
      concat_ws(
        "/",
        collect_set(
          case
            when
              QuoteStatus in ("Bound", "Processing Outstanding")
              and QuoteOption = "bind"
              and isincluded = true
            then
              mLOB.Line_Display_Name
            else null
          end
        ) OVER (PARTITION BY fPCD.InsuredGUID, dD.ddFiscalYear)
      )
    else NULL
  end as FY_Product_Combination_Per_Insured,
  case
    when
      Quote_Status in ("Bound", "Processing Outstanding")
      and QuoteOption = "bind"
      and isincluded = true
    then
      concat_ws(
        "/",
        collect_set(
          case
            when
              QuoteStatus in ("Bound", "Processing Outstanding")
              and QuoteOption = "bind"
              and isincluded = true
            then
              fPCD.Binder_Group
            else null
          end
        ) OVER (PARTITION BY fPCD.InsuredGUID)
      )
    else NULL
  end as Overall_Binder_Combination_Per_Insured,
  case
    when
      Quote_Status in ("Bound", "Processing Outstanding")
      and QuoteOption = "bind"
      and isincluded = true
    then
      concat_ws(
        "/",
        collect_set(
          case
            when
              QuoteStatus in ("Bound", "Processing Outstanding")
              and QuoteOption = "bind"
              and isincluded = true
            then
              fPCD.Binder_Group
            else null
          end
        ) OVER (PARTITION BY fPCD.InsuredGUID, dD.ddFiscalYear)
      )
    else NULL
  end as FY_Binder_Combination_Per_Insured,
  MAX(RevenueBasis) OVER (
      PARTITION BY fPCD.InsuredGUID, dD.ddFiscalYear
    ) as FY_Turnover_Per_Insured,
  fPCD.RiskLimit As Risk_Limit,
  fPCD.NumberOfEmployees As Number_Of_Employees,
  fPCD.RevenueBasis As Turnover,
  fPCD.Deductible As Excess,
  fPCD.RenewalHandling As Renewal_Handling,
  fPCD.RenewalComment As Renewal_Comment,
  fPCD.ProducerCompanyCode As Broker_Code,
  Case
    When
      fPCD.TransactionTypeID Is Null
      OR fPCD.TransactionTypeID = "ORIGNAL PREMIUM"
    Then
      "Original"
    When fPCD.TransactionTypeID IN ("E", "N", "R") Then "MTA"
    When fPCD.TransactionTypeID = "C" Then "Cancellation"
    Else "MTA"
  End As Policy_Transaction_Type,
  Case
    When
      fPCD.QuoteInitiatedBy <> "BizCover"
      And CAST(
        CONCAT(
          YEAR(Coalesce(fPCD.VarStartDate, fPCD.StartDate)),
          LPAD(MONTH(Coalesce(fPCD.VarStartDate, fPCD.StartDate)), 2, "0")
        ) AS INT
      )
        < CAST(
          CONCAT(
            YEAR(fPCD.GreaterOfEffectiveOrBindDate),
            LPAD(MONTH(fPCD.GreaterOfEffectiveOrBindDate), 2, "0")
          ) AS INT
        )
    Then
      "Yes"
    Else "No"
  End As Late_Processing_Flag,
  fPCD.UnderwriterNameFirstLast As Underwriter_Name,
  CAST(date_format(fPCD.StartDate, "yyyy-MM-dd") AS DATE) As Policy_Inception_Date,
  CAST(date_format(fPCD.ExpiryDate, "yyyy-MM-dd") AS DATE) As Policy_Expiry_Date,
  CAST(
    date_format(Coalesce(fPCD.EndorsementEffectiveDate, DATE("1900-01-01")), "yyyy-MM-dd") AS DATE
  ) As Policy_Endorsement_Effective_Date,
  CAST(date_format(fPCD.ExpiryDate, "yyyy-MM-dd") AS DATE) As Policy_Endorsement_Expiry_Date,
  Coalesce(fPCD.BaseCurrencyOwnPremium, 0) As Gross_Written_Premium_Amount,
  Coalesce(fPCD.DualAdminFee, 0) As Admin_Fee_Amount,
  try_divide(
    COALESCE(fPCD.ProducerCommissionAmount, 0),
    Coalesce(fPCD.BaseCurrencyOwnPremium, 0)
  ) As Producing_Brokerage_Percentage,
  Coalesce(fPCD.ProducerCommissionAmount, 0) As Brokerage_on_Premium_Amount,
  Coalesce(fPCD.NetWrittenPremium, 0) As Net_Written_Premium_Amount,
  case
    when
      fpcd.Datasource = "eGlobal"
      and fPCD.GreaterOfEffectiveOrBindDate >= "2024-06-01"
      and fPCD.GreaterOfEffectiveOrBindDate <= "2025-05-31"
      and fPCD.PlacingBrokerName1 like "Bowood%"
    then
      (Coalesce(fPCD.NewTotalCedeCommission, fPCD.TotalCedeCommission, 0))
        + (0.0075 * Coalesce(fPCD.BaseCurrencyOwnPremium, 0))
    else COALESCE(NULLIF(Coalesce(fPCD.NewTotalCedeCommission, 0), 0), fPCD.TotalCedeCommission)
  end As Cede_Commission_Amount,
  case
    when
      fpcd.Datasource = "eGlobal"
      and fPCD.GreaterOfEffectiveOrBindDate >= "2024-06-01"
      and fPCD.GreaterOfEffectiveOrBindDate <= "2025-05-31"
      and fPCD.PlacingBrokerName1 like "Bowood%"
    then
      (Coalesce(Coalesce(fPCD.NewTotalCedeCommission, fPCD.TotalCedeCommission), 0))
        + (0.0075 * Coalesce(fPCD.BaseCurrencyOwnPremium, 0))
        + Coalesce(fPCD.DualAdminFee, 0)
    else
      Coalesce(Coalesce(fPCD.NewTotalCedeCommission, fPCD.TotalCedeCommission), 0)
        + Coalesce(fPCD.DualAdminFee, 0)
  end As Dual_Income,
  fPCD.IndustryActivityCodeDescription As Activity_Code_Description,
  fPCD.BinderYear As Year_Letter,
  fPCD.BinderReference As Binder_Name,
  fPCD.BinderSection As Binder_Section,
  fPCD.BinderCode As Binder_Code,
  fPCD.BinderUMR As UMR,
  fPCD.Binder_Group as Binder_Group,
  fPCD.Contract,
  Case
    WHEN fPCD.QuoteStatus <> "Bound" THEN "Expiring Policy is Cancelled"
    WHEN
      (
        fPCD.isRunoff = true
        OR fPCD.runoffIsDiscovery = true
      )
    THEN
      "Expiring Policy is in Run-Off"
    WHEN fPCD.RenewalHandling = "NonRenewable" THEN "Marked as NonRenewable"
    WHEN fPCD.RenewalHandling = "Declined" THEN "Marked as Declined"
    ELSE "Renewable"
  End As Renewable_Status,
  Case
    When
      (
        fPCD.isRunoff = true
        OR fPCD.runoffIsDiscovery = true
        OR fPCD.QuoteStatus <> "Bound"
        OR fPCD.RenewalHandling = "NonRenewable"
        OR fPCD.RenewalHandling = "Declined"
      )
    Then
      "No"
    Else "Yes"
  End As Renewable_Flag,
  Case
    When
      (
        fPCD.isRunoff = true
        OR fPCD.runoffIsDiscovery = true
      )
    Then
      "Yes"
    Else "No"
  End As Discovery_Period_RunOff_Applied_Flag,
  Case
    When fPCD.DataSource in ("Trak", "NZ BO", "eGlobal") Then "Y"
    When
      fPCD.DataSource = "IMS" ----Jira DMO-837 ml fPCD.DataSource IN ("IMS")
      and fPCD.BoundDate is NOT NULL
      and fPCD.QuoteOption = "bind"
      and fPCD.isincluded = true
    Then
      "Y"
    Else "N"
  End As Premium_Flag,
  Case
    When
      fPCD.DataSource IN ("IMS") --Jira DMO-837 ml fPCD.DataSource IN ("IMS") need to add eGlobal
      And fPCD.LatestBoundQuotePerPolicy = fPCD.QuoteID
      and fPCD.QuoteOption = "bind"
      and fPCD.isincluded = true
      And fPCD.RiskID IS NOT NULL
    Then
      "Y"
    Else "N"
  End As IMS_Latest_Bound_Quote_Flag,
  Case
    When
      fPCD.DataSource IN ("IMS") --Jira DMO-837 ml fPCD.DataSource IN ("IMS") need to add eGlobal
      And fPCD.EarliestQuotePerPolicy = fPCD.QuoteID
      and fPCD.QuoteOption = "bind"
      and fPCD.isincluded = true
      And fPCD.RiskID IS NOT NULL
    Then
      "Y"
    Else "N"
  End As IMS_Earliest_Bound_Quote_Flag,
  TRIM(
    concat(
      mLOB.Line_Name,
      "|",
      Coalesce(mLOB.Line_Subtype, ""),
      "|",
      Coalesce(
        case
          when mLOB.Scheme_Name = "Bizcover" then "BizCover"
          when mLOB.Scheme_Name = "Abacus - Aon" then "Abacus - AON"
          when mLOB.Scheme_Name = "LPMS - Aon" then "LPMS - AON"
          else mLOB.Scheme_Name
        end,
        ""
      ),
      "|",
      Coalesce(mLoB.Data_Source, ""),
      "|",
      mLOB.Country_Office,
      "|",
      case
        when mLoB.Data_Source = "NZ BO" then mLOB.Line_Display_Name
        --when mLOB.Line_Name="Natural Disaster" then mLOB.Line_Display_Name  ---#Buydoun ND test
        else "N"
      end,
      "|",
      case
        when mLoB.Data_Source = "NZ BO" then mLOB.Detailed_Line_Display_Name
        else "N"
      end
    )
  ) As Product_Key,
  case
    when
      fPCD.UnderwriterNameFirstLast like "BizCover%"
      And fPCD.DataSource = "IMS"
      And fPCD.isincluded = true
      And fPCD.QuoteOption = "bind"
      And fPCD.BoundDate is NOT NULL
    then
      "Y"
    Else "N"
  end as BizCover_IMS_Flag,
  fPCD.externalTransactionReference As BizCover_Transaction_Id,
  IFF(
    fPCD.DataSource = "IMS"
    and (
      fPCD.QuoteID = Coalesce(fPCD.LatestBoundQuotePerPolicy, 0)
      And (
        fPCD.QuoteOption = "bind"
        And fPCD.isincluded = true
      )
    ),
    "Y",
    IFF(
      fPCD.DataSource = "IMS"
      and fPCD.QuoteID = Coalesce(fPCD.LatestBoundQuotePerPolicy, fPCD.LatestQuotePerPolicy),
      "Y",
      "N"
    )
  ) as IMS_Portfolio_Flag,
  case
    when
      fPCD.RiskId is null
    then
      Row_Number() OVER (
          PARTITION BY fPCD.DataSource, fPCD.ControlNo, fPCD.LineName, fPCD.LineKey
          ORDER BY fPCD.QuoteID desc
        )
    else 1
  end as Control_No_Row_Id,
  Case
    When
      fPCD.DataSource IN ("IMS")--ml , "eGlobal"
      And fPCD.LatestBoundQuotePerPolicy = fPCD.QuoteID
      And fPCD.RiskID IS NOT NULL
    then
      Renewing_Quote.RenewingControlNo
    else NULL
  end as Renewing_Control_No,
  Case
    When
      fPCD.DataSource IN ("IMS") --, "eGlobal"
      And fPCD.LatestBoundQuotePerPolicy = fPCD.QuoteID
      And fPCD.RiskID IS NOT NULL
    then
      Renewing_Quote.RenewingQuoteStatus
    else NULL
  end as Renewing_Quote_Status,
  Case
    When
      fPCD.DataSource IN ("IMS")--ml , "eGlobal"
      And fPCD.LatestBoundQuotePerPolicy = fPCD.QuoteID
      And fPCD.RiskID IS NOT NULL
    then
      Renewing_Quote.RenewingQuoteId
    else NULL
  end as Renewing_Quote_Id,
  Case
    When
      fPCD.DataSource IN ("IMS")--ml, "eGlobal"
      And fPCD.LatestBoundQuotePerPolicy = fPCD.QuoteID
      And fPCD.RiskID IS NOT NULL
    then
      Renewing_Quote.RenewingGrossWrittenPremiumAmount
    else NULL
  end as Renewing_Gross_Written_Premium_Amount,
  Case
    When
      fPCD.DataSource IN ("IMS")----ml, "eGlobal"
      And fPCD.LatestBoundQuotePerPolicy = fPCD.QuoteID
      And fPCD.RiskID IS NOT NULL
    then
      Renewing_Quote.TotalRenewingGrossWrittenPremiumAmount
    else NULL
  end as Renewing_Total_Gross_Written_Premium_Amount,
  Case
    When
      fPCD.DataSource IN ("IMS")--ml, "eGlobal"
      And fPCD.LatestBoundQuotePerPolicy = fPCD.QuoteID
      And fPCD.RiskID IS NOT NULL
    then
      Renewing_Quote.TotalOriginalRenewingGrossWrittenPremiumAmount
    else NULL
  end as Total_Original_Renewing_Gross_Written_Premium_Amount,
  Case
    When
      fPCD.DataSource IN ("IMS")--ml, "eGlobal"
      And fPCD.LatestBoundQuotePerPolicy = fPCD.QuoteID
      And fPCD.RiskID IS NOT NULL
    then
      Renewing_Quote.TotalMTARenewingGrossWrittenPremiumAmount
    else NULL
  end as Total_MTA_Renewing_Gross_Written_Premium_Amount,
  Case
    When
      fPCD.DataSource IN ("IMS")--ml, "eGlobal"
      And fPCD.LatestBoundQuotePerPolicy = fPCD.QuoteID
      And fPCD.RiskID IS NOT NULL
    then
      Renewing_Quote.TotalRenewingCedeCommissionAmount
    else NULL
  end as Renewing_Total_Cede_Commission_Amount,
  Case
    When
      fPCD.DataSource IN ("IMS")--ml, "eGlobal"
      And fPCD.LatestBoundQuotePerPolicy = fPCD.QuoteID
      And fPCD.RiskID IS NOT NULL
    then
      Renewing_Quote.TotalRenewingAdminFeeAmount
    else NULL
  end as Renewing_Total_Admin_Fee_Amount,
  Case
    When
      fPCD.DataSource IN ("IMS")--ml, "eGlobal"
      And fPCD.LatestBoundQuotePerPolicy = fPCD.QuoteID
      And fPCD.RiskID IS NOT NULL
    then
      Renewing_Quote.RenewingBrokerName
    else NULL
  end as Renewing_Broker_Name,
  Case
    When
      fPCD.DataSource IN ("IMS")--ml, "eGlobal"
      And fPCD.LatestBoundQuotePerPolicy = fPCD.QuoteID
      And fPCD.RiskID IS NOT NULL
    then
      Renewing_Quote.RenewingUnderwriterName
    else NULL
  end as Renewing_Underwriter_Name,
  Case
    When
      fPCD.DataSource IN ("IMS")--ml, "eGlobal"
      And fPCD.LatestBoundQuotePerPolicy = fPCD.QuoteID
      And fPCD.RiskID IS NOT NULL
    then
      Coalesce(
        Nullif(Renewing_Quote.TotalBoundRenewingGrossWrittenPremiumAmount, 0),
        Case
          When fPCD.LineName = "Property" Then coalesce(fPCD.ExpiringPremium, 0)
          Else 0
        End
      )
    else NULL
  end as Total_Bound_Renewing_Gross_Written_Premium_Amount,
  Case
    When
      fPCD.DataSource IN ("IMS")--ml, "eGlobal"
      And fPCD.LatestBoundQuotePerPolicy = fPCD.QuoteID
      And fPCD.RiskID IS NOT NULL
    then
      Renewing_Quote.TotalCancelledRenewingGrossWrittenPremiumAmount
    else NULL
  end as Total_Cancelled_Renewing_Gross_Written_Premium_Amount,
  CONCAT(
    fPCD.RiskID,
    "|",
    TRIM(
      concat(
        mLOB.Line_Name,
        "|",
        Coalesce(mLOB.Line_Subtype, ""),
        "|",
        Coalesce(
          case
            when mLOB.Scheme_Name = "Bizcover" then "BizCover"
            else mLOB.Scheme_Name
          end,
          ""
        ),
        "|",
        Coalesce(mLoB.Data_Source, ""),
        "|",
        mLOB.Country_Office
      )
    )
  ) as Policy_Key,
  Policy_Totals.gwp as Total_Gross_Written_Premium_Amount,
  Policy_Totals.bound_gwp as Total_Bound_Gross_Written_Premium_Amount,
  Policy_Totals.cancelled_gwp as Total_Cancelled_Gross_Written_Premium_Amount,
  Policy_Totals.cede as Total_Cede_Commission_Amount,
  Policy_Totals.adminfee as Total_Admin_Fee_Amount,
  Policy_Totals.income as Total_Dual_Income,
  Policy_Totals.brokerage as Total_Brokerage,
  case
    when fPCD.ProducerRegion = "NSW" then "Northern"
    else fPCD.ProducerRegion
  end as Producer_Region,
  CASE
    WHEN fpcd.BaseCurrencyOwnPremium <= 1000 THEN CONCAT("1. ", CHR(36), "0", " - ", CHR(36), "1K")
    WHEN
      fpcd.BaseCurrencyOwnPremium <= 1000
      AND QuoteStatus in (
        "Lapsed", "Declined", "Blank", "Not Taken Up", "Lost", "Indicated", "Referred", "Incomplete"
      )
    THEN
      CONCAT("1. ", CHR(36), "0", " - ", CHR(36), "1K")
    WHEN
      fpcd.BaseCurrencyOwnPremium <= 2500
    THEN
      CONCAT("2. ", CHR(36), "1K", " - ", CHR(36), "2.5K")
    WHEN
      fpcd.BaseCurrencyOwnPremium <= 5000
    THEN
      CONCAT("3. ", CHR(36), "2.5K", " - ", CHR(36), "5K")
    WHEN
      fpcd.BaseCurrencyOwnPremium <= 10000
    THEN
      CONCAT("4. ", CHR(36), "5K", " - ", CHR(36), "10K")
    WHEN
      fpcd.BaseCurrencyOwnPremium <= 20000
    THEN
      CONCAT("5. ", CHR(36), "10K", " - ", CHR(36), "20K")
    WHEN
      fpcd.BaseCurrencyOwnPremium <= 50000
    THEN
      CONCAT("6. ", CHR(36), "20K", " - ", CHR(36), "50K")
    WHEN fpcd.BaseCurrencyOwnPremium IS NULL THEN "8. Missing GWP Amount"
    ELSE CONCAT("7. Greater than ", CHR(36), "50K")
  END AS Premium_Banding,
  coalesce(
    fPCD.ExpiringPremium,
    Expiring_Totals.gwp
  ) as Expiring_Total_Gross_Written_Premium_Amount,
  ----ml Jira DMO-746 revert back
  /*CASE
    WHEN
      Coinsurance_isCoinsurance = true
      and mLOB.Product_Group not like "%Property%"
    THEN
      coalesce(fPCD.ExpiringPremium, Expiring_Totals.gwp) * Coinsurance_participationPercentage
    ELSE coalesce(fPCD.ExpiringPremium, Expiring_Totals.gwp)
  END AS Expiring_Total_Gross_Written_Premium_Amount,*/
  fPCD.UserEmail as Underwriter_Email,
  fPCD.ProducerContactEmail as Broker_Contact_Email,
  fPCD.ProducerContactName as Broker_Contact_Name,
  IFF(
    fPCD.DataSource = "IMS"
    and (
      fPCD.QuoteID = Coalesce(fPCD.LatestBoundQuotePerPolicy, 0)
      And (
        fPCD.QuoteOption = "bind"
        And fPCD.isincluded = true
      )
    ),
    fPCD.TechPremium,
    IFF(
      fPCD.DataSource = "IMS"
      and fPCD.QuoteID = Coalesce(fPCD.LatestBoundQuotePerPolicy, fPCD.LatestQuotePerPolicy),
      fPCD.TechPremium,
      null
    )
  ) as Tech_Premium_Amount,
  Case
    When
      fPCD.DataSource = "IMS"
      And fPCD.LatestBoundQuotePerPolicy = fPCD.QuoteID
      And fPCD.RiskID IS NOT NULL
    Then
      fPCD.TechPremium
    Else null
  End As Latest_Bound_Tech_Premium_Amount,
  Renewing_Quote.RenewingTechPremiumAmount as Renewing_Tech_Premium_Amount,
  case
    when
      Renewing_Quote.RenewingQuoteStatus in ("Bound", "Processing Outstanding")
    then
      Renewing_Quote.RenewingTechPremiumAmount
    else null
  end as Latest_Bound_Renewing_Tech_Premium_Amount,
  IFF(
    fPCD.DataSource = "IMS"
    and (
      fPCD.QuoteID = Coalesce(fPCD.LatestBoundQuotePerPolicy, 0)
      And (
        fPCD.QuoteOption = "bind"
        And fPCD.isincluded = true
      )
    ),
    fPCD.RevenueBasis,
    IFF(
      fPCD.DataSource = "IMS"
      and fPCD.QuoteID = Coalesce(fPCD.LatestBoundQuotePerPolicy, fPCD.LatestQuotePerPolicy),
      fPCD.RevenueBasis,
      null
    )
  ) as Latest_Turnover,
  Case
    When
      fPCD.DataSource = "IMS"
      And fPCD.LatestBoundQuotePerPolicy = fPCD.QuoteID
      And fPCD.RiskID IS NOT NULL
    Then
      fPCD.RevenueBasis
    Else null
  End As Latest_Bound_Turnover,
  Renewing_Quote.RenewingTurnover as Latest_Renewing_Turnover,
  case
    when
      Renewing_Quote.RenewingQuoteStatus in ("Bound", "Processing Outstanding")
    then
      Renewing_Quote.RenewingTurnover
    else null
  end as Latest_Bound_Renewing_Turnover,
  IFF(
    fPCD.DataSource = "IMS"
    and (
      fPCD.QuoteID = Coalesce(fPCD.LatestBoundQuotePerPolicy, 0)
      And (
        fPCD.QuoteOption = "bind"
        And fPCD.isincluded = true
      )
    ),
    fPCD.Deductible,
    IFF(
      fPCD.DataSource = "IMS"
      and fPCD.QuoteID = Coalesce(fPCD.LatestBoundQuotePerPolicy, fPCD.LatestQuotePerPolicy),
      fPCD.Deductible,
      null
    )
  ) as Latest_Excess,
  Case
    When
      fPCD.DataSource = "IMS"
      And fPCD.LatestBoundQuotePerPolicy = fPCD.QuoteID
      And fPCD.RiskID IS NOT NULL
    Then
      fPCD.Deductible
    Else null
  End As Latest_Bound_Excess,
  Renewing_Quote.RenewingExcess as Latest_Renewing_Excess,
  case
    when
      Renewing_Quote.RenewingQuoteStatus in ("Bound", "Processing Outstanding")
    then
      Renewing_Quote.RenewingExcess
    else null
  end as Latest_Bound_Renewing_Excess,
  IFF(
    fPCD.DataSource = "IMS"
    and (
      fPCD.QuoteID = Coalesce(fPCD.LatestBoundQuotePerPolicy, 0)
      And (
        fPCD.QuoteOption = "bind"
        And fPCD.isincluded = true
      )
    ),
    fPCD.RiskLimit,
    IFF(
      fPCD.DataSource = "IMS"
      and fPCD.QuoteID = Coalesce(fPCD.LatestBoundQuotePerPolicy, fPCD.LatestQuotePerPolicy),
      fPCD.RiskLimit,
      null
    )
  ) as Latest_Risk_Limit,
  Case
    When
      fPCD.DataSource = "IMS"
      And fPCD.LatestBoundQuotePerPolicy = fPCD.QuoteID
      And fPCD.RiskID IS NOT NULL
    Then
      fPCD.RiskLimit
    Else null
  End As Latest_Bound_Risk_Limit,
  Renewing_Quote.RenewingRiskLimit as Latest_Renewing_Risk_Limit,
  case
    when
      Renewing_Quote.RenewingQuoteStatus in ("Bound", "Processing Outstanding")
    then
      Renewing_Quote.RenewingRiskLimit
    else null
  end as Latest_Bound_Renewing_Risk_Limit,
  fPCD.PolicyProductWording as Policy_Product_Wording,
  fPCD.BusinessType as Business_Type,
  fPCD.CoverageClass as Coverage_Class,
  1 as row_id,
  fPCD.ProgramId as Program_Id,
  --Jira: DMO-741 --case when fPCD.QuoteValidUntil > Expiring_Totals.latestexpirydate then "Yes" else "No" end as Hold_Cover_Flag,
  case
    when
      date_format(cast(QuoteValidUntil AS date), "yyyy-MM")
        > date_format(cast(ExpiryDate AS date), "yyyy-MM")
    then
      "Yes"
    else "No"
  end Hold_Cover_Flag,
  Expiring_Totals.latestexpirydate as Previous_Policy_Expiry_Date,
  DATEDIFF(fPCD.QuoteValidUntil, Expiring_Totals.latestexpirydate) as Quote_Valid_Until_Days,
  DATEDIFF(fPCD.BoundDate, CAST(fPCD.WebRaterEntryDate as DATE)) as Quote_Submission_to_Bound_Days,
  DATEDIFF(fPCD.BoundDate, Expiring_Totals.latestexpirydate) as Expiring_to_Renewing_Bound_Days,
  fPCD.Coinsurance_participationPercentage as IUA_Proportion_of_Risk,
  fPCD.TransactionTypeID as Transaction_Type_Id,
  fPCD.risktransactionType as Risk_Transaction_Type,
  fPCD.QuoteInitiatedBy,
  --fPCD.firstIndicatedBy,
  --fPCD.firstQuotedBy,
  fPCD.FENZGST as FENZ_GST,
  fPCD.FENZAmount as FENZ_Amount,
  fPCD.NHCAmount as NHC_Amount,
  fPCD.NHCGST as NHC_GST,
  case
    when
      fpcd.Linename = "Professional Indemnity"
      and fpcd.DataSource = "IMS"
    then
      concat(
        mlob.Detailed_Line_Display_Name,
        "-",
        CASE
          WHEN fpcd.PolicyProductWording ILIKE "%Accountants%" THEN "Accountants"
          WHEN
            fpcd.PolicyProductWording ILIKE "%Design and Engineering%"
          THEN
            "Design and Engineering"
          WHEN fpcd.PolicyProductWording ILIKE "%Design and Construct%" THEN "Design and Construct"
          WHEN fpcd.PolicyProductWording ILIKE "%Real Estate%" THEN "Real Estate"
          WHEN
            fpcd.PolicyProductWording ILIKE "%Financial Consultants%"
          THEN
            "Financial Consultants"
          WHEN fpcd.PolicyProductWording ILIKE "%Consultants%" THEN "Consultants"
          WHEN fpcd.PolicyProductWording ILIKE "%Solicitors%" THEN "Solicitors"
          ELSE "Other"
        end
      )
    else mlob.Detailed_Line_Display_Name
  end as PI_Sub_Product,
  fPCD.IndustryActivityCode as Activity_Code,
  fPCD.Coinsurance_isCoinsurance as Is_Coinsurance,
  fPCD.Coinsurance_Role as Coinsurance_Role,
  fPCD.ActivityIndustryDescriptionGroup as Activity_Industry_Description_Group,
  fPCD.PIActivityIndustryGroup as PI_Rating_Industry_Group,
  fPCD.DistributionChannelTypeOfInsurance as BizCover_Distribution_Channel_Type,
  fPCD.DistributionChannelName as BizCover_Distribution_Channel,
  fPCD.BIZCoverIntermediateBrokerName as BizCover_Intermediated_Brokerage_Name
from
  fPCD
    LEFT JOIN ' ||env_var|| '.brz.tbl_RiskWriteDistributionChannel RWDC
      ON RWDC.UniquePolicyReference
        = CONCAT(
          fPCD.RiskID,
          " | ",
          fPCD.LineSubtype,
          " | ",
          fPCD.LineName,
          " | ",
          fPCD.BinderReference
        )
    LEFT JOIN ' ||env_var|| '.gld.vw_dim_product mLOB  --testing buydown excess 
      ON (
        case
          when fPCD.Datasource = "NZ BO" then upper(fPCD.Contract)
          else fPCD.LineName
        end
      ) = mLOB.Line_Name
      And case
        when fPCD.Datasource = "NZ BO" then UPPER(Coalesce(fPCD.LineSubtype, ""))
        else Coalesce(fPCD.LineSubtype, "")
      end = mLOB.Line_Subtype
      And case
        when fPCD.Datasource = "NZ BO" then UPPER(Coalesce(fPCD.SchemeName, ""))
        else Coalesce(fPCD.SchemeName, "")
      end = mLOB.Scheme_Name
      And fPCD.DataSource = mLoB.Data_Source
      And fPCD.CompanyISOCountryCode = mLOB.Country_Office
    LEFT JOIN Renewing_Quote
      on fPCD.RiskID = Renewing_Quote.PreviousPolicyNumber
      and fPCD.LineID = Renewing_Quote.LineID
      and fPCD.LineSubtype = Renewing_Quote.LineSubtype
    LEFT JOIN Policy_Totals
      on fPCD.ControlNo = Policy_Totals.ControlNo
      and fPCD.LineID = Policy_Totals.LineID
      and fPCD.LineSubtype = Policy_Totals.LineSubtype
    LEFT JOIN Expiring_Totals
      on fPCD.PreviousPolicyNumber = Expiring_Totals.RiskID
      and fPCD.LineID = Expiring_Totals.LineID
      and fPCD.LineSubtype = Expiring_Totals.LineSubtype
    LEFT JOIN ' ||env_var|| '.brz.dimdate dD
      on fPCD.GreaterOfEffectiveOrBindDate = dD.ddYearMonthDay_DateOnly
UNION ALL
select
  *
from
  (
    Select
      fPCD_Allianz.DataSource As Data_Source,
      cast(DATEADD(YEAR, -1, fPCD_Allianz.GreaterOfEffectiveOrBindDate) as date) As Reporting_Date,
      cast(DATEADD(YEAR, -1, fpCD_Allianz.QuoteValidUntil) as date) as Quote_Valid_Until_Date,
      CAST(
        date_format(DATEADD(YEAR, -1, fPCD_Allianz.BoundDate), "yyyy-MM-dd") AS DATE
      ) as Quote_Submission_Date,
      fPCD_Allianz.PlacingBroker as Placing_Broker,
      fPCD_Allianz.CurrencyCode as Currency_Code,
      fPCD_Allianz.QuoteGUID as Quote_Guid,
      fPCD_Allianz.QuoteOption as Quote_Option,
      Concat(
        Case fPCD_Allianz.CompanyISOCountryCode
          When "AUS" Then "AU"
          Else "NZ"
        End,
        "|",
        fPCD_Allianz.ProducerCompanyCode
      ) as Broker_Key,
      fPCD_Allianz.SchemeName /* KK - Added this as per DMO-815 */,
      CASE
        WHEN fPCD_Allianz.ProducerCompanyCode = "BIZN" THEN "BizCover"
        WHEN
          fPCD_Allianz.QuoteInitiatedBy = "IMS"
          AND Coalesce(fPCD_Allianz.SchemeName, "") not in (
            "AON - Non Aligned Advisers", "AON Combined Advisor", "Penberthy", "Penberthy Platinum"
          )
        THEN
          "Specialty"
        WHEN
          fPCD_Allianz.QuoteInitiatedBy = "PORTAL"
          and Coalesce(fPCD_Allianz.SchemeName, "") not in (
            "AON - Non Aligned Advisers", "AON Combined Advisor", "Penberthy", "Penberthy Platinum"
          )
        THEN
          "WebRater"
        WHEN fPCD_Allianz.QuoteInitiatedBy = "BIZCOVER" THEN "BizCover"
        WHEN
          fPCD_Allianz.QuoteInitiatedBy = "Trak"
          and RWDC.DistributionChannel IS NOT NULL
        THEN
          REPLACE(RWDC.DistributionChannel, "WebRater - Rules", "WebRater")
        WHEN
          fPCD_Allianz.QuoteInitiatedBy = "Trak"
          and RWDC.DistributionChannel IS NULL
          and fPCD_Allianz.ProducerCompanyCode in ("ACSV", "SURN", "AIPM", "CMIS", "PMIS", "PRRS")
        THEN
          "Scheme"
        WHEN
          fPCD_Allianz.DataSource = "IMS"
          and fPCD_Allianz.CompanyISOCountryCode = "NZL"
          and fPCD_Allianz.SchemeName in (
            "AON - Non Aligned Advisers", "AON Combined Advisor", "Penberthy", "Penberthy Platinum"
          )
        THEN
          "Scheme"
        
        WHEN fPCD_Allianz.ClientName  ilike  "ACS Scheme%"  --("ACS Scheme - Schools","ACS Scheme - Churches")
        THEN
          "Scheme"
        WHEN
          fPCD_Allianz.SchemeName in (
            "Abacus - AON",
            "Abacus - Aon",
            "Advisorcover - MA Risk",
            "AMP Advisers - AON",
            "Aon-CEAS",
            "Architects and Graphic Designers - AIB",
            "ATF - Marsh",
            "Body Corporate - Crombie Lockwood",
            "Bookkeepers - AON",
            "Brokerweb - Willis",
            "Bureau - Churches Scheme",
            "Churches Scheme - Bureau",
            "CoffeeSure",
            "Counter Cover",
            "Cyber Offer",
            "EPA - Marsh",
            "Fabworx - CBA",
            "Ginger Group - AON",
            "Grocer Guard - Rothbury",
            "Harcourts - Crombie Lockwood",
            "HRINZ - Rothbury",
            "IAA & NZAMI - APEX",
            "ICIB Travel Agents - Marsh",
            "iiTP - I2I Brokers",
            "Kiwibank Contractors - i2ibrokers",
            "Law Plus - Willis",
            "Licensed Building Practitioners - Marsh",
            "Lockton - Inbroke",
            "LPMS - AON",
            "LPMS - Aon",
            "Marsh Accountants",
            "Marsh Big Accountants",
            "Marsh Small Accountants",
            "Mortgage Link - AON",
            "Non-aligned Advisers - Aon",
            "NorthSurance",
            "NZACS",
            "NZACS - Aon",
            "NZFMPF - Non Aligned - AON",
            "NZFMPS - AON",
            "NZFMSA",
            "NZFOA & NZFFA - AON Forests",
            "NZIBS - Willis",
            "Photographers - Rothbury",
            "Probus New Zealand Corporate Travel - Willis",
            "Project Arrow - Marsh",
            "PSC - Interflora",
            "Real Estate",
            "Real Estate - AON",
            "Real Estate - Penberthy",
            "Rebus New Zealand Corporate Travel",
            "Rebus New Zealand Corporate Travel - Willis",
            "REINZ - Crombie Lockwood",
            "Restaurants - PSC",
            "Roofers Association - Austinsure",
            "Sage Partners - Forestry",
            "Scheme",
            "Schools - Apex",
            "Valuers Inspectors - JLT",
            "Video Retailers",
            "Videoguard - Hutchinson Rodway",
            "Vision Sure",
            "AON - Non Aligned Advisers",
            "AON Combined Advisor",
            "PDA Professional Indemnity" /* KK - Added this one and below two schemes as per DMO-805 */,
            "PDA Information Technology",
            "PDA Management Liability"
          )
        THEN
          "Scheme"
        ELSE "Specialty"
      END As Segment,
      CAST(
        date_format(DATEADD(YEAR, -1, fPCD_Allianz.BoundDate), "yyyy-MM-dd") AS DATE
      ) As Bound_Date,
      CAST(
        date_format(DATEADD(YEAR, -1, fPCD_Allianz.CreatedDate), "yyyy-MM-dd") AS DATE
      ) As Sales_Date,
      CONCAT(
        CAST(fPCD_Allianz.controlno As String),
        "X""_",
        SUBSTRING(
          fPCD_Allianz.TransactionID,
          CHARINDEX("_", fPCD_Allianz.TransactionID) + 1,
          LEN(fPCD_Allianz.TransactionID)
        )
      ) As Transaction_Id,
      NULL As Control_No,
      PreviousPolicyNumber As Policy_Number,
      RIGHT(CAST(fPCD_Allianz.QuoteID As String), 3) As Quote_Id,
      concat(
        RIGHT(CAST(fPCD_Allianz.QuoteGUID As String), 3),
        "|",
        fPCD_Allianz.lineproducttype
      ) as Quote_Product_Key,
      "Bound" As Quote_Status,
      NULL as Quote_Status_Reason,
      CASE
        WHEN fPCD_Allianz.PolicyTypeID = 1 Then "New Business"
        WHEN fPCD_Allianz.policyTypeId = 2 THEN "Renewal"
        ELSE cast(fPCD_Allianz.policyTypeId As String)
      END As Overarching_Policy_Type,
      Case
        When
          fPCD_Allianz.DataSource = "IMS"
        Then
          Case
            When Coalesce(fPCD_Allianz.CurrentlyInsuredWith, "") = "DUAL Renewal" Then "Renewal"
            Else "New Business"
          End
        When
          fPCD_Allianz.DataSource = "Trak"
        Then
          CASE
            WHEN fPCD_Allianz.PolicyTypeID = 1 Then "New Business"
            WHEN fPCD_Allianz.PolicyTypeId = 2 THEN "Renewal"
            ELSE cast(fPCD_Allianz.policyTypeId As String)
          END
      End As Policy_Type,
      NULL As Previous_Policy_Number,
      CASE
        WHEN
          CHARINDEX(";", fPCD_Allianz.ClientName) > 0
        THEN
          LEFT(fPCD_Allianz.ClientName, CHARINDEX(";", fPCD_Allianz.ClientName) - 1)
        WHEN
          CHARINDEX(",", fPCD_Allianz.ClientName) > 0
        THEN
          LEFT(fPCD_Allianz.ClientName, CHARINDEX(",", fPCD_Allianz.ClientName) - 1)
        ELSE ClientName
      END AS Client_Name,
      NULL as Insured_Id,
      NULL as Overall_Product_Combination_Per_Insured,
      NULL as FY_Product_Combination_Per_Insured,
      NULL as Overall_Binder_Combination_Per_Insured,
      NULL as FY_Binder_Combination_Per_Insured,
      NULL as FY_Turnover_Per_Insured,
      fPCD_Allianz.RiskLimit As Risk_Limit,
      fPCD_Allianz.NumberOfEmployees As Number_Of_Employees,
      fPCD_Allianz.RevenueBasis As Revenue_Basis,
      fPCD_Allianz.Deductible As Deductible,
      fPCD_Allianz.RenewalHandling As Renewal_Handling,
      fPCD_Allianz.RenewalComment As Renewal_Comment,
      fPCD_Allianz.ProducerCompanyCode As Broker_Code,
      "Original" As Policy_Transaction_Type,
      "No" As Late_Processing_Flag,
      fPCD_Allianz.UnderwriterNameFirstLast As Underwriter_Name,
      CAST(
        date_format(add_months(fPCD_Allianz.StartDate, -12), "yyyy-MM-dd") AS DATE
      ) As Policy_Inception_Date,
      CAST(date_format(fPCD_Allianz.StartDate, "yyyy-MM-dd") AS DATE) As Policy_Expiry_Date,
      CAST(
        date_format(
          Coalesce(add_months(fPCD_Allianz.EndorsementEffectiveDate, -12), DATE("1900-01-01")),
          "yyyy-MM-dd"
        ) AS DATE
      ) As Policy_Endorsement_Effective_Date,
      CAST(
        date_format(fPCD_Allianz.StartDate, "yyyy-MM-dd") AS DATE
      ) As Policy_Endorsement_Expiry_Date,
      0 As Gross_Written_Premium_Amount,
      0 As Admin_Fee_Amount,
      0 As Producing_Brokerage_Percentage,
      0 As Brokerage_on_Premium_Amount,
      0 As Net_Written_Premium_Amount,
      0 As Cede_Commission_Amount,
      0 As Dual_Income,
      fPCD_Allianz.IndustryActivityCodeDescription As Activity_Code_Description,
      fPCD_Allianz.BinderYear As Year_Letter,
      fPCD_Allianz.BinderReference As Binder_Name,
      fPCD_Allianz.BinderSection As Binder_Section,
      fPCD_Allianz.BinderCode As Binder_Code,
      fPCD_Allianz.BinderUMR As UMR,
      NULL as Binder_Group,
      fPCD_Allianz.Contract,
      "Renewable" As Renewable_Status,
      "Yes" As Renewable_Flag,
      NULL As Discovery_Period_RunOff_Applied_Flag,
      "Y" As Premium_Flag,
      "Y" As IMS_Latest_Bound_Quote_Flag,
      "Y" as IMS_Earliest_Bound_Quote_Flag,
      TRIM(
        concat(
          mLOB.Line_Name,
          "|",
          Coalesce(mLOB.Line_Subtype, ""),
          "|",
          Coalesce(
            case
              when mLOB.Scheme_Name = "Bizcover" then "BizCover"
              else mLOB.Scheme_Name
            end,
            ""
          ),
          "|",
          Coalesce(mLoB.Data_Source, ""),
          "|",
          mLOB.Country_Office,
          "|",
          "N",
          "|",
          "N"
        )
      ) As Product_Key,
      "N" as BizCover_IMS_Flag,
      NULL As BizCover_Transaction_Id,
      "Y" as IMS_Portfolio_Flag,
      1 as Control_No_Row_Id,
      ControlNo as Renewing_Control_No,
      fPCD_Allianz.QuoteStatus as Renewing_Quote_Status,
      QuoteID as Renewing_Quote_Id,
      Coalesce(fPCD_Allianz.BaseCurrencyOwnPremium, 0) as Renewing_Gross_Written_Premium_Amount,
      SUM(
        case
          when
            fPCD_Allianz.QuoteStatus in (
              "Bound", "Processing Outstanding", "Cancelled", "Notice of Cancellation"
            )
            and fPCD_Allianz.QuoteOption = "bind"
            and fPCD_Allianz.isincluded = true
          then
            Coalesce(BaseCurrencyOwnPremium, 0)
          else 0
        end
      ) OVER (
          PARTITION BY
            fPCD_Allianz.previouspolicynumber,
            fPCD_Allianz.LineId,
            fPCD_Allianz.Linesubtype
          order by fPCD_Allianz.startdate
        ) as Renewing_Total_Gross_Written_Premium_Amount,
      SUM(
        case
          when
            fPCD_Allianz.QuoteStatus in (
              "Bound", "Processing Outstanding", "Cancelled", "Notice of Cancellation"
            )
            and fPCD_Allianz.QuoteOption = "bind"
            and fPCD_Allianz.isincluded = true
            and (
              TransactionTypeID Is Null
              OR TransactionTypeID = "ORIGNAL PREMIUM"
              OR TransactionTypeID = "C"
            )
          then
            Coalesce(BaseCurrencyOwnPremium, 0)
          else 0
        end
      ) OVER (
          PARTITION BY
            fPCD_Allianz.previouspolicynumber,
            fPCD_Allianz.LineId,
            fPCD_Allianz.Linesubtype
          order by fPCD_Allianz.startdate
        ) as Total_Original_Renewing_Gross_Written_Premium_Amount,
      SUM(
        case
          when
            fPCD_Allianz.QuoteStatus in (
              "Bound", "Processing Outstanding", "Cancelled", "Notice of Cancellation"
            )
            and fPCD_Allianz.QuoteOption = "bind"
            and fPCD_Allianz.isincluded = true
            and TransactionTypeID IN ("E", "N", "R")
          then
            Coalesce(BaseCurrencyOwnPremium, 0)
          else 0
        end
      ) OVER (
          PARTITION BY
            fPCD_Allianz.previouspolicynumber,
            fPCD_Allianz.LineId,
            fPCD_Allianz.Linesubtype
          order by fPCD_Allianz.startdate
        ) as Total_MTA_Renewing_Gross_Written_Premium_Amount,
      SUM(
        case
          when
            fPCD_Allianz.QuoteStatus in (
              "Bound", "Processing Outstanding", "Cancelled", "Notice of Cancellation"
            )
            and fPCD_Allianz.QuoteOption = "bind"
            and fPCD_Allianz.isincluded = true
          then
            Coalesce(Coalesce(NewTotalCedeCommission, TotalCedeCommission), 0)
          else 0
        end
      ) OVER (
          PARTITION BY
            fPCD_Allianz.previouspolicynumber,
            fPCD_Allianz.LineId,
            fPCD_Allianz.Linesubtype
          order by fPCD_Allianz.startdate
        ) as Renewing_Total_Cede_Commission_Amount,
      SUM(
        case
          when
            fPCD_Allianz.QuoteStatus in (
              "Bound", "Processing Outstanding", "Cancelled", "Notice of Cancellation"
            )
            and fPCD_Allianz.QuoteOption = "bind"
            and fPCD_Allianz.isincluded = true
          then
            Coalesce(DualAdminFee, 0)
          else 0
        end
      ) OVER (
          PARTITION BY
            fPCD_Allianz.previouspolicynumber,
            fPCD_Allianz.LineId,
            fPCD_Allianz.Linesubtype
          order by fPCD_Allianz.startdate
        ) as Renewing_Total_Admin_Fee_Amount,
      fPCD_Allianz.ProducerCompanyName as Renewing_Broker_Name,
      UnderwriterNameFirstLast as Renewing_Underwriter_Name,
      SUM(
        case
          when
            fPCD_Allianz.QuoteStatus in ("Bound", "Processing Outstanding")
            and fPCD_Allianz.QuoteOption = "bind"
            and fPCD_Allianz.isincluded = true
          then
            Coalesce(BaseCurrencyOwnPremium, 0)
          else 0
        end
      ) OVER (
          PARTITION BY
            fPCD_Allianz.previouspolicynumber,
            fPCD_Allianz.LineId,
            fPCD_Allianz.Linesubtype
          order by fPCD_Allianz.startdate
        ) as Total_Bound_Renewing_Gross_Written_Premium_Amount,
      SUM(
        case
          when
            fPCD_Allianz.QuoteStatus in ("Cancelled", "Notice of Cancellation")
          then
            Coalesce(BaseCurrencyOwnPremium, 0)
          else 0
        end
      ) OVER (
          PARTITION BY
            fPCD_Allianz.previouspolicynumber,
            fPCD_Allianz.LineId,
            fPCD_Allianz.Linesubtype
          order by fPCD_Allianz.startdate
        ) as Total_Cancelled_Renewing_Gross_Written_Premium_Amount,
      NULL as Policy_Key,
      ExpiringPremium as Total_Gross_Written_Premium_Amount,
      ExpiringPremium as Total_Bound_Gross_Written_Premium_Amount,
      0 as Total_Cancelled_Gross_Written_Premium_Amount,
      0 as Total_Cede_Commission_Amount,
      0 as Total_Admin_Fee_Amount,
      0 as Total_Dual_Income,
      0 as Total_Brokerage,
      case
        when fPCD_Allianz.ProducerRegion = "NSW" then "Northern"
        else fPCD_Allianz.ProducerRegion
      end as Producer_Region,
      CONCAT("1. ", CHR(36), "0", " - ", CHR(36), "1K") Premium_Banding,
      0 as Expiring_Total_Gross_Written_Premium_Amount,
      fPCD_Allianz.UserEmail as Under_writer_Email,
      fPCD_Allianz.ProducerContactEmail as Broker_Contact_Email,
      fPCD_Allianz.ProducerContactName as Broker_Contact_Name,
      FIRST_VALUE(fPCD_Allianz.TechPremium) OVER (
          PARTITION BY fPCD_Allianz.ControlNo
          ORDER BY fPCD_Allianz.QuoteID DESC
        ) AS Tech_Premium_Amount,
      FIRST_VALUE(fPCD_Allianz.TechPremium) OVER (
          PARTITION BY fPCD_Allianz.ControlNo
          ORDER BY fPCD_Allianz.QuoteID DESC
        ) As Latest_Bound_Tech_Premium_Amount,
      FIRST_VALUE(fPCD_Allianz.TechPremium) OVER (
          PARTITION BY fPCD_Allianz.ControlNo
          ORDER BY fPCD_Allianz.QuoteID DESC
        ) as Renewing_Tech_Premium_Amount,
      case
        when
          FIRST_VALUE(fPCD_Allianz.QuoteStatus) OVER (
              PARTITION BY fPCD_Allianz.ControlNo
              ORDER BY fPCD_Allianz.QuoteID DESC
            ) in ("Bound", "Processing Outstanding")
        then
          FIRST_VALUE(fPCD_Allianz.TechPremium) OVER (
              PARTITION BY fPCD_Allianz.ControlNo
              ORDER BY fPCD_Allianz.QuoteID DESC
            )
        else null
      end as Latest_Bound_Renewing_Tech_Premium_Amount,
      FIRST_VALUE(fPCD_Allianz.RevenueBasis) OVER (
          PARTITION BY fPCD_Allianz.ControlNo
          ORDER BY fPCD_Allianz.QuoteID DESC
        ) AS Latest_Turnover,
      FIRST_VALUE(fPCD_Allianz.RevenueBasis) OVER (
          PARTITION BY fPCD_Allianz.ControlNo
          ORDER BY fPCD_Allianz.QuoteID DESC
        ) As Latest_Bound_Turnover,
      FIRST_VALUE(fPCD_Allianz.RevenueBasis) OVER (
          PARTITION BY fPCD_Allianz.ControlNo
          ORDER BY fPCD_Allianz.QuoteID DESC
        ) as Latest_Renewing_Turnover,
      case
        when
          FIRST_VALUE(fPCD_Allianz.QuoteStatus) OVER (
              PARTITION BY fPCD_Allianz.ControlNo
              ORDER BY fPCD_Allianz.QuoteID DESC
            ) in ("Bound", "Processing Outstanding")
        then
          FIRST_VALUE(fPCD_Allianz.RevenueBasis) OVER (
              PARTITION BY fPCD_Allianz.ControlNo
              ORDER BY fPCD_Allianz.QuoteID DESC
            )
        else null
      end as Latest_Bound_Renewing_Turnover,
      FIRST_VALUE(fPCD_Allianz.Deductible) OVER (
          PARTITION BY fPCD_Allianz.ControlNo
          ORDER BY fPCD_Allianz.QuoteID DESC
        ) AS Latest_Excess,
      FIRST_VALUE(fPCD_Allianz.Deductible) OVER (
          PARTITION BY fPCD_Allianz.ControlNo
          ORDER BY fPCD_Allianz.QuoteID DESC
        ) As Latest_Bound_Excess,
      FIRST_VALUE(fPCD_Allianz.Deductible) OVER (
          PARTITION BY fPCD_Allianz.ControlNo
          ORDER BY fPCD_Allianz.QuoteID DESC
        ) as Latest_Renewing_Excess,
      case
        when
          FIRST_VALUE(fPCD_Allianz.QuoteStatus) OVER (
              PARTITION BY fPCD_Allianz.ControlNo
              ORDER BY fPCD_Allianz.QuoteID DESC
            ) in ("Bound", "Processing Outstanding")
        then
          FIRST_VALUE(fPCD_Allianz.Deductible) OVER (
              PARTITION BY fPCD_Allianz.ControlNo
              ORDER BY fPCD_Allianz.QuoteID DESC
            )
        else null
      end as Latest_Bound_Renewing_Excess,
      FIRST_VALUE(fPCD_Allianz.RiskLimit) OVER (
          PARTITION BY fPCD_Allianz.ControlNo
          ORDER BY fPCD_Allianz.QuoteID DESC
        ) AS Latest_Risk_Limit,
      FIRST_VALUE(fPCD_Allianz.RiskLimit) OVER (
          PARTITION BY fPCD_Allianz.ControlNo
          ORDER BY fPCD_Allianz.QuoteID DESC
        ) As Latest_Bound_Risk_Limit,
      FIRST_VALUE(fPCD_Allianz.RiskLimit) OVER (
          PARTITION BY fPCD_Allianz.ControlNo
          ORDER BY fPCD_Allianz.QuoteID DESC
        ) as Latest_Renewing_Risk_limit,
      case
        when
          FIRST_VALUE(fPCD_Allianz.QuoteStatus) OVER (
              PARTITION BY fPCD_Allianz.ControlNo
              ORDER BY fPCD_Allianz.QuoteID DESC
            ) in ("Bound", "Processing Outstanding")
        then
          FIRST_VALUE(fPCD_Allianz.RiskLimit) OVER (
              PARTITION BY fPCD_Allianz.ControlNo
              ORDER BY fPCD_Allianz.QuoteID DESC
            )
        else null
      end as Latest_Bound_Renewing_Risk_Limit,
      fPCD_Allianz.PolicyProductWording as Policy_Product_Wording,
      fPCD_Allianz.BusinessType as Business_Type,
      fPCD_Allianz.CoverageClass as Coverage_Class,
      fPCD_Allianz.row_id,
      fPCD_Allianz.ProgramId as Program_Id,
      "No" as Hold_Cover_Flag,
      CAST(
        date_format(add_months(fPCD_Allianz.StartDate, -24), "yyyy-MM-dd") AS DATE
      ) as Previous_Policy_Expiry_Date,
      0 as Quote_Valid_Until_Days,
      0 as Quote_Submission_to_Bound_Days,
      0 as Expiring_to_Renewing_Bound_Days,
      fPCD_Allianz.Coinsurance_participationPercentage as IUA_Proportion_of_Risk,
      fPCD_Allianz.TransactionTypeID as Transaction_Type_Id,
      fPCD_Allianz.risktransactionType as Risk_Transaction_Type,
      fPCD_Allianz.QuoteInitiatedBy,
      --fPCD_Allianz.firstIndicatedBy,
      --fPCD_Allianz.firstQuotedBy,
      fPCD_Allianz.FENZGST as FENZ_GST,
      fPCD_Allianz.FENZAmount as FENZ_Amount,
      fPCD_Allianz.NHCAmount as NHC_Amount,
      fPCD_Allianz.NHCGST as NHC_GST,
      mlob.Detailed_Line_Display_Name as PI_Sub_Product,
      fPCD_Allianz.IndustryActivityCode as Activity_Code,
      fPCD_Allianz.Coinsurance_isCoinsurance as Is_Coinsurance,
      fPCD_Allianz.Coinsurance_Role as Coinsurance_Role,
      fPCD_Allianz.ActivityIndustryDescriptionGroup as Activity_Industry_Description_Group,
      fPCD_Allianz.PIActivityIndustryGroup as PI_Rating_Industry_Group,
      fPCD_Allianz.DistributionChannelTypeOfInsurance as BizCover_Distribution_Channel_Type,
      fPCD_Allianz.DistributionChannelName as BizCover_Distribution_Channel,
      fPCD_Allianz.BIZCoverIntermediateBrokerName as BizCover_Intermediated_Brokerage_Name
    from
      fPCD_Allianz
        LEFT JOIN ' ||env_var|| '.brz.tbl_RiskWriteDistributionChannel RWDC
          ON RWDC.UniquePolicyReference
            = CONCAT(
              fPCD_Allianz.RiskID,
              " | ",
              fPCD_Allianz.LineSubtype,
              " | ",
              fPCD_Allianz.LineName,
              " | ",
              fPCD_Allianz.BinderReference
            )
        LEFT JOIN ' ||env_var|| '.gld.vw_dim_product mLOB
          ON fPCD_Allianz.Linename = mLOB.Line_Name
          And Coalesce(fPCD_Allianz.LineSubtype, "") = Coalesce(mLOB.Line_Subtype, "")
          And Coalesce(fPCD_Allianz.SchemeName, "") = Coalesce(mLOB.Scheme_Name, "")
          And fPCD_Allianz.DataSource = mLoB.Data_Source
          And fPCD_Allianz.CompanyISOCountryCode = mLOB.Country_Office
       
  ) Allianz
where
  Allianz.row_id = 1'